In [1]:
# CELL 1: Environment & Thread Locks
import os
import torch

os.environ["CACHE_LOOKBACK"] = "41"
os.environ["CACHE_START_DATE"] = "1998-01-01"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

# Prevent sub-process thread explosion across CPU cores
torch.set_num_threads(1)

In [2]:
# CELL 3: Path Setup
import sys
from pathlib import Path

RUNNING_IN_COLAB = "google.colab" in sys.modules
if RUNNING_IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    rl_root = Path(
        "/content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2"
    )
else:
    rl_root = Path("C:/Users/ping/Files_win10/python/py311/stocks/notebooks_RLVR_v2")

if str(rl_root) not in sys.path:
    sys.path.append(str(rl_root))
os.chdir(rl_root)

In [3]:
# CELL 4: Imports
import torch
import numpy as np
import pandas as pd
from core.settings import TradingConfig, CacheConfig
from core.paths import OUTPUT_DIR, LOCAL_DATA_DIR
from data_pipeline.loader import load_processed_data
from data_pipeline.utils import get_master_trading_calendar, get_chronological_splits
from data_pipeline.screener import UniverseScreener
from rl_discovery.oracle import RLOracle
from rl_discovery.environment import DiscoveryEnv
from rl_discovery.adapter import RLVRGymEnv, ObservationScaler
from rl_discovery.agent import AbsoluteZeroAgent
from rl_discovery.trainer import RolloutBuffer, PPOTrainer
from rl_discovery.validator import AgentEvaluator

NOTEBOOKS_RLVR_ROOT: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2
PROJECT_ROOT: C:\Users\ping\Files_win10\python\py311\stocks
GLOBAL_DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\data
GLOBAL_PROCESSED_DIR: C:\Users\ping\Files_win10\python\py311\stocks\data\processed
LOCAL_DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\data
OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\output



In [4]:
# CELL 5: Load Data & Memory-Optimized Environment Splits
import gc

print("Loading preprocessed data & AlphaCache...")
df_ohlcv, macro_df, features_df = load_processed_data()
config = TradingConfig()
df_close = df_ohlcv["Adj Close"].unstack(level=0).sort_index()
trading_calendar = get_master_trading_calendar(df_ohlcv, config.calendar_ticker)

screener = UniverseScreener(
    df_close=df_close,
    features_df=features_df,
    macro_df=macro_df,
    trading_calendar=trading_calendar,
    config=config,
)

cache_file_path = LOCAL_DATA_DIR / CacheConfig.get_filename()
print(f"cache_file_path:\n{cache_file_path}\n")
feature_cube = pd.read_parquet(cache_file_path)

# Forward 1-Day MTM Simple Return Matrix (P_{t+1} - P_t) / P_t
reward_matrix = df_close.pct_change(1).shift(-1)
reward_matrix["CASH"] = 0.0

# -------------------------------------------------------------------
# EXPLICIT TRAIN / VAL / TEST SPLITS
# -------------------------------------------------------------------
valid_dates = feature_cube.index.get_level_values("Date").unique()
valid_calendar = trading_calendar[trading_calendar.isin(valid_dates)]

VAL_START = pd.Timestamp("2016-01-01")
TEST_START = pd.Timestamp("2022-04-01")

cal_train_raw = valid_calendar[valid_calendar < VAL_START]
cal_val_raw = valid_calendar[
    (valid_calendar >= VAL_START) & (valid_calendar < TEST_START)
]
cal_test_raw = valid_calendar[valid_calendar >= TEST_START]

# Prevent Data Leakage (Holding Period Buffer)
hp = config.holding_period
cal_train = cal_train_raw[:-hp] if len(cal_train_raw) > hp else cal_train_raw
cal_val = cal_val_raw[:-hp] if len(cal_val_raw) > hp else cal_val_raw
cal_test = cal_test_raw

print(f"Custom splits applied! Leakage buffer dropped: {hp} days.")

# -------------------------------------------------------------------
# RAM DEFENSE: Free massive raw DataFrames not needed for training
# -------------------------------------------------------------------
# del df_ohlcv
# gc.collect()
# print("🧹 Freed raw OHLCV and Feature tables from RAM.")

# -------------------------------------------------------------
# ENVIRONMENT INITIALIZATIONS (Dynamic 46-Obs / 16-Action)
# -------------------------------------------------------------
env_train = DiscoveryEnv(
    feature_cube=feature_cube,
    reward_matrix=reward_matrix,
    calendar=cal_train,
    macro_df=macro_df,
    config=config,
)
gym_train = RLVRGymEnv(env_train, macro_df)

env_val = DiscoveryEnv(
    feature_cube=feature_cube,
    reward_matrix=reward_matrix,
    calendar=cal_val,
    macro_df=macro_df,
    config=config,
)
gym_val = RLVRGymEnv(env_val, macro_df)

env_test = DiscoveryEnv(
    feature_cube=feature_cube,
    reward_matrix=reward_matrix,
    calendar=cal_test,
    macro_df=macro_df,
    config=config,
)
gym_test = RLVRGymEnv(env_test, macro_df)

# Freeze scalers for Validation and Test environments
gym_val.is_training = False
gym_test.is_training = False

Loading preprocessed data & AlphaCache...
cache_file_path:
C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2\data\alpha_cache_41d_1998.parquet



C:\Users\ping\AppData\Local\Temp\ipykernel_13924\2338645649.py:23: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  reward_matrix = df_close.pct_change(1).shift(-1)


Custom splits applied! Leakage buffer dropped: 5 days.


In [5]:
print(f"df_ohlcv.info():\n{df_ohlcv.info()}")

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 8570001 entries, ('A', Timestamp('1999-11-18 00:00:00')) to ('ZWS', Timestamp('2026-08-19 00:00:00'))
Data columns (total 5 columns):
 #   Column     Dtype  
---  ------     -----  
 0   Adj Open   float64
 1   Adj High   float64
 2   Adj Low    float64
 3   Adj Close  float64
 4   Volume     float64
dtypes: float64(5)
memory usage: 360.0+ MB
df_ohlcv.info():
None


In [6]:
print(f"macro_df.info():\n{macro_df.info()}")

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 7454 entries, 1997-01-02 to 2026-08-19
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Mkt_Ret              7454 non-null   float64
 1   Mkt_Ret_Z            7454 non-null   float64
 2   Macro_Trend          7454 non-null   float64
 3   Macro_Trend_Z        7454 non-null   float64
 4   Yield_Curve_10Y2Y_Z  7454 non-null   float64
 5   Macro_Trend_Vel_Z    7454 non-null   float64
 6   Macro_Trend_Mom      7454 non-null   float64
 7   Macro_Vix_Z          7454 non-null   float64
 8   Macro_Vix_Ratio      7454 non-null   float32
 9   Mkt_Vol_63d_Z        7454 non-null   float64
dtypes: float32(1), float64(9)
memory usage: 869.5 KB
macro_df.info():
None


In [7]:
print(f"features_df.info():\n{features_df.info()}")

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 8570001 entries, ('A', Timestamp('1999-11-18 00:00:00')) to ('ZWS', Timestamp('2026-08-19 00:00:00'))
Data columns (total 22 columns):
 #   Column               Dtype  
---  ------               -----  
 0   ATR                  float64
 1   ATRP                 float64
 2   TRP                  float64
 3   RSI                  float64
 4   Mom_21               float64
 5   Consistency          float64
 6   IR_63                float64
 7   Beta_63              float64
 8   DD_21                float64
 9   AutoCorr_15          float64
 10  Ret_1d               float64
 11  Range_Pos_20         float64
 12  Slope_P_5            float64
 13  Slope_V_5            float64
 14  Slope_P_5_Z          float64
 15  Slope_V_5_Z          float64
 16  Convexity            float64
 17  RollingStalePct      float64
 18  RollMedDollarVol     float64
 19  RollingSameVolCount  float64
 20  RecentStaleDays      float64
 21  IsZeroPrice          int64  

In [8]:
print(f"df_close.info():\n{df_close.info()}")

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 7454 entries, 1997-01-02 to 2026-08-19
Columns: 1716 entries, A to ZWS
dtypes: float64(1716)
memory usage: 97.9 MB
df_close.info():
None


In [9]:
print(f"feature_cube.info():\n{feature_cube.info()}")

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 5050917 entries, (Timestamp('1998-01-02 00:00:00'), 'AA') to (Timestamp('2026-08-19 00:00:00'), 'ZTS')
Data columns (total 12 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   41d_Log Price Gain          float64
 1   41d_Sharpe (TRP)            float64
 2   41d_Momentum (21d)          float64
 3   41d_Info Ratio (63d)        float64
 4   41d_Oversold (-RSI)         float64
 5   41d_Dip Buyer (-dd_21)      float64
 6   41d_Range Position (20d)    float64
 7   41d_Return Autocorr (15d)   float64
 8   41d_Low Volatility (-ATRP)  float64
 9   41d_Slope_P_5_Z             float64
 10  41d_Slope_V_5_Z             float64
 11  41d_Convexity               float64
dtypes: float64(12)
memory usage: 482.0+ MB
feature_cube.info():
None


In [10]:
df_ohlcv.loc["QQQ"]

,Adj Open,Adj High,Adj Low,Adj Close,Volume
Date,,,,,
1999-03-10,43.0797,43.1061,42.3688,43.0271,6209091.0
1999-03-11,43.3430,43.5932,42.3951,43.2377,11497976.0
1999-03-12,43.0797,43.1060,41.8421,42.1844,10376496.0
1999-03-15,42.5004,43.4484,42.0528,43.3957,7558429.0
1999-03-16,43.5800,43.9487,43.1061,43.7644,5821972.0
...,...,...,...,...,...
2026-08-13,725.1500,733.9600,724.0300,732.0700,31599700.0
2026-08-14,733.4100,734.3900,728.3200,731.0700,23790600.0
2026-08-17,732.9500,734.5800,729.2700,729.8700,26452600.0
